In [ ]:
from pathlib import Path

import altair as alt
import polars as pl
import polars.selectors as cs

Caveats (from Downtown Travel Study; did not look at 2018 BATS data for this exercise):

- delivery-food includes to any location in 2018, vs only to home in 2022
- survey instrument in 2018 is not explicit about whether all the other delivery\_\*
  variables include food deliveries, so we just assume they don't since delivery_food
  is a variable in itself
- for some reason the 2022 dataset distinguishes between packages and 'items'
- 2022 has a variable for 'Received packages at another location (e.g., Amazon Locker,
  package pick-up point)', whereas the corresponding variable for 2018 is just
  'Received package/delivery to a locker on travel date'. There's also a 'Received other
  type of delivery on travel date' in 2018 (though only 136 entries / 2226 person-days
  in the entire Bay Area). I decided to merge all of these into a single
  'delivery-other' variable.


In [ ]:
# whether to include data from day == 5, 6, 7 using pdexpfac from day == 1, 2, 3, 4
include_weekends = True

In [ ]:
data_raw_dir = Path(
    r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Deliverable_20241127"
)
data_processed_dir = Path(
    r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\tdm_market_analysis"
)

In [ ]:
counties = {
    6075: "sf",
    6041: "ex_sf",  # "marin",
    6097: "ex_sf",  # "sonoma",
    6055: "ex_sf",  # "napa",
    6095: "ex_sf",  # "solano",
    6013: "ex_sf",  # "contra_costa",
    6001: "ex_sf",  # "alameda",
    6085: "ex_sf",  # "santa_clara",
    6081: "ex_sf",  # "san_mateo",
}

In [ ]:
def load_data(data_raw_dir, data_processed_dir):
    personday = (
        (  # get pdexpfac from pipeline results
            pl.read_csv(
                data_processed_dir / "personday.csv",
                columns=["hhno", "pno", "day", "pdexpfac"],
            )
        )
        .join(  # get delivery columns from raw
            pl.read_csv(
                data_raw_dir / "day.csv",
                columns=[
                    "hh_id",
                    "person_num",
                    "travel_dow",
                    "delivery_2",
                    # drop 3: Someone came to do work at home (e.g., babysitter, housecleaning, lawn)
                    # "delivery_3",
                    "delivery_4",
                    "delivery_5",
                    "delivery_6",
                    "delivery_7",
                    "delivery_8",
                    "delivery_9",
                    "delivery_996",
                ],
            ).rename({"hh_id": "hhno", "person_num": "pno", "travel_dow": "day"}),
            on=["hhno", "pno", "day"],
            how="left",
        )
        .join(
            # load hh table to get home_county (renamed to hh_geog)
            pl.read_csv(data_raw_dir / "hh.csv", columns=["hh_id", "home_county"])
            .with_columns(pl.col("home_county").replace_strict(counties))
            .rename({"hh_id": "hhno", "home_county": "hh_geog"}),
            on=["hhno"],
            how="left",
        )
        .join(
            pl.read_csv(
                data_processed_dir / "person.csv",
                # separator=" ",
                columns=["hhno", "pno", "pagey"],  # "hhtaz", "pwtaz"
            ),
            on=["hhno", "pno"],
            how="left",  # keep persondays, just get age #, hhtaz, pwtaz
        )
    )
    return personday


def handle_weekends(personday, include_weekends):
    # logic to include weekends (day == 5, 6, 7) using pdexpfac from day == 1, 2, 3, 4

    # check that pdexpfac is unique for each hhno x pno
    # (pdexpfac is 0 when day == 5, 6, or 7)
    assert (
        personday.filter(pl.col("day").is_in([1, 2, 3, 4]))
        .group_by(["hhno", "pno"])
        .agg(pl.col("pdexpfac").n_unique())  # should be 1
        .filter(pl.col("pdexpfac") > 1)
        .is_empty()
    )
    if include_weekends:  # include day == 5, 6, 7 using pdexpfac from day == 1, 2, 3, 4
        personday = personday.with_columns(
            pdexpfac=pl.when(pl.col("day").is_in([5, 6, 7]))
            .then(
                pl.col("pdexpfac")
                .filter(pl.col("day").is_in([1, 2, 3, 4]))
                .first()
                .over(["hhno", "pno"])
            )
            .otherwise(pl.col("pdexpfac"))
        )
    return personday

In [ ]:
personday_2022 = (
    load_data(data_raw_dir, data_processed_dir)
    .filter(pl.col("pagey") > 17)  # adults only, to be consistent with 2019
    .with_columns(
        cs.starts_with("delivery").replace_strict({1: True, 0: False, 995: None})
    )
    .rename(
        {
            "delivery_2": "delivery_home_food",
            "delivery_4": "delivery_home_groceries",
            "delivery_5": "delivery_home_packages",
            "delivery_6": "delivery_work_personalpackages",
            # 7: # another location (e.g., Amazon Locker, package pick-up point)
            "delivery_7": "delivery_locker_packages",
            # 8: Other items delivered to home (e.g., appliance)
            "delivery_8": "delivery_home_other",  # exc packages
            "delivery_9": "delivery_work_other",  # exc personal packages
            "delivery_996": "delivery_none",
        }
    )
    .select(  # rename for easier comparison between 2018/2022
        "hhno",
        "pno",
        "day",
        "hh_geog",
        # "pw_geog",
        "pdexpfac",
        "delivery_none",
        "delivery_home_food",
        "delivery_home_groceries",
        "delivery_home_packages",
        "delivery_work_personalpackages",
        "delivery_locker_packages",
        "delivery_home_other",
        "delivery_work_other",
    )
)
personday_2022 = personday_2022.with_columns(
    # set persondays for which respondent says they got deliveries but not in any
    # delivery type (i.e. no delivery variable is True) to missing response / null
    # (this is still logically possible in 2018 given survey instrument wording,
    # but is impossible under 2022 survey instrument wording)
    **{
        col: pl.when(
            ~(
                pl.col("delivery_none")
                | pl.col("delivery_home_food")
                | pl.col("delivery_home_groceries")
                | pl.col("delivery_home_packages")
                | pl.col("delivery_work_personalpackages")
                | pl.col("delivery_locker_packages")
                | pl.col("delivery_home_other")
                | pl.col("delivery_work_other")
            )
        )
        .then(pl.lit(None))
        .otherwise(pl.col(col))
        for col in personday_2022.select(cs.starts_with("delivery")).columns
    }
)

personday_2022 = handle_weekends(personday_2022, include_weekends)

personday_2022_rough = (
    personday_2022.select(  # rename to have uniform variable names between 2018/2022
        "hhno",
        "pno",
        "day",
        "hh_geog",
        # "pw_geog",
        "pdexpfac",
        "delivery_none",
        delivery_food=pl.col("delivery_home_food"),  # only includes to home!
        delivery_home=(
            pl.col("delivery_home_groceries")
            | pl.col("delivery_home_packages")
            | pl.col("delivery_home_other")
        ),
        delivery_work=(
            pl.col("delivery_work_personalpackages") | pl.col("delivery_work_other")
        ),
        # actually for all non work/home locations
        # "another location (e.g., Amazon Locker, package pick-up point)""
        delivery_other=pl.col("delivery_locker_packages"),
    ).with_columns(
        delivery_any=(~pl.col("delivery_none")),
        delivery_foodhomeother=(
            pl.col("delivery_food") | pl.col("delivery_home") | pl.col("delivery_other")
        ),
    )
)

personday_2022_fine = (
    personday_2022.select(
        "hhno",
        "pno",
        "day",
        "hh_geog",
        # "pw_geog",
        "pdexpfac",
        "delivery_none",
        "delivery_home_groceries",
        "delivery_home_packages",
        "delivery_home_other",
        "delivery_work_personalpackages",
        "delivery_work_other",
        # decided to keep food under home for the 'fine' categories because we're not
        # doing the nested bar graph anyways
        "delivery_home_food",
        # delivery_food=pl.col("delivery_home_food"),  # only includes to home!
        delivery_other_packages=pl.col("delivery_locker_packages"),  # actually locker
        delivery_home_multicats=(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            > 1
        ),
        delivery_home_2cats=pl.when(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            == 2
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False)),
        delivery_home_3cats=pl.when(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            == 3
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False)),
        delivery_home_4cats=pl.when(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            == 4
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False)),
        delivery_work_2cats=(
            pl.col("delivery_work_personalpackages") & pl.col("delivery_work_other")
        ),
    )
    .with_columns(
        delivery_home_food=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_food")),
        delivery_home_groceries=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_groceries")),
        delivery_home_packages=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_packages")),
        delivery_home_other=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_other")),
        delivery_work_personalpackages=pl.when("delivery_work_2cats")
        .then(False)
        .otherwise(pl.col("delivery_work_personalpackages")),
        delivery_work_other=pl.when("delivery_work_2cats")
        .then(False)
        .otherwise(pl.col("delivery_work_other")),
    )
    .drop("delivery_home_multicats")
)

In [ ]:
personday_2022

In [ ]:
print("sample sizes after excluding missing responses")
(
    personday_2022.with_columns(missing_response=pl.col("delivery_none").is_null())
    .filter(~pl.col("missing_response"))
    .group_by("hh_geog")
    .agg(
        hh_unexpanded=pl.n_unique("hhno"),
        person_unexpanded=pl.struct("hhno", "pno").n_unique(),
        personday_unexpanded=pl.len(),
        personday_expanded=pl.sum("pdexpfac"),
    )
)

In [ ]:
print("make sure that when delivery-none is true, there are indeed no deliveries")
display(
    personday_2022.filter(pl.col("delivery_none"))
    .select(cs.starts_with("delivery"))
    .unique()
)

In [ ]:
def calc_sums(personday, geog):
    return (
        personday.with_columns(missing_response=pl.col("delivery_none").is_null())
        .group_by(geog)
        .agg(
            **{
                col: pl.col("pdexpfac").filter(col).sum()
                for col in (
                    personday.select(cs.starts_with("delivery")).columns
                    + ["missing_response"]
                )
            },
            total=pl.sum("pdexpfac"),
            total_exc_missingresponse=pl.col("pdexpfac")
            .filter(~pl.col("missing_response"))
            .sum(),
        )
        .sort(geog)
    )


def add_shares(df, col, sum_over_cols="year"):
    return df.with_columns(
        (pl.col("weighted") / pl.col("weighted").sum().over(sum_over_cols)).alias(
            f"{col}-share"
        )
    )


def plot_dist(df, df_name, col):
    # df = (
    #     df.group_by(col).agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort(col)
    # )
    df = df.with_columns(pl.col("delivery_type").replace({"missing_response": None}))
    return df, alt.Chart(df.drop_nulls()).mark_bar().encode(
        x=alt.X(f"{col}:O", title=None, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("weighted", title=col),
        color=color_value_transit_teal,
    ).properties(title=f"{df_name}")
    # + df.plot.bar(
    #     x=col, y="unweighted", title=f"{df_name}\n{col}: unweighted", frame_width=200
    # ).opts(xrotation=45)


def plot_dists(df2018, df2022, df_name, col):
    df2018, plot2018 = plot_dist(df2018, f"{df_name} preCOVID", col)
    df2022, plot2022 = plot_dist(df2022, f"{df_name} postCOVID", col)
    display(df2018)
    display(df2022)
    df = pl.concat(
        [
            df2018.with_columns(year=pl.lit(2019)),
            df2022.with_columns(year=pl.lit(2023)),
        ]
    )
    print("look here to get null shares (non-response rate):")
    display(add_shares(df.drop_nulls(), col))
    df = df.drop_nulls()
    print("sums 2018/2022", df2018.sum(), df2022.sum())
    return df, (plot2018 | plot2022).resolve_scale(y="shared")

In [ ]:
# home location
delivery_home_2022_rough_wide = calc_sums(personday_2022_rough, "hh_geog")
delivery_home_2022_fine_wide = calc_sums(personday_2022_fine, "hh_geog")
# work location
# delivery_work_2022_rough_wide = calc_sums(personday_2022_rough, "pw_geog")
# delivery_work_2022_fine_wide = calc_sums(personday_2022_fine, "pw_geog")

In [ ]:
delivery_home_2022_rough_wide

In [ ]:
delivery_home_2022_rough_wide

In [ ]:
delivery_home_2022_rough_wide.select("hh_geog", "total_exc_missingresponse")

In [ ]:
delivery_home_2022_rough_long = (
    delivery_home_2022_rough_wide.unpivot(
        index=["hh_geog"],
        on=[
            "delivery_none",
            "delivery_any",
            "delivery_food",
            "delivery_home",
            # Downtown Today report not interested in deliveries to work location
            # "delivery_work",
            "delivery_other",
            # drop missing responses to calculate per personday rates
            # "missing_response",
        ],
        variable_name="delivery_type",
        value_name="weighted",
    )
    .join(
        delivery_home_2022_rough_wide.select("hh_geog", "total_exc_missingresponse"),
        on="hh_geog",
    )
    .with_columns(
        weighted_per_hhgeog_personday=(
            pl.col("weighted") / pl.col("total_exc_missingresponse")
        )
    )
    .drop("total_exc_missingresponse")
)
delivery_home_2022_fine_long = add_shares(
    (
        delivery_home_2022_fine_wide.unpivot(
            index=["hh_geog"],
            on=[
                "delivery_none",
                "delivery_other_packages",
                "delivery_home_food",
                "delivery_home_groceries",
                "delivery_home_packages",
                "delivery_home_other",
                "delivery_home_2cats",
                "delivery_home_3cats",
                "delivery_home_4cats",
                # Downtown Today report not interested in deliveries to work location
                # "delivery_work_personalpackages",
                # "delivery_work_other",
                # "delivery_work_2cats",
                # drop missing responses to calculate per personday rates
                # and for calculating shares
                # "missing_response",
            ],
            variable_name="delivery_type",
            value_name="weighted",
        )
        .with_columns(year=pl.lit(2023))
        .join(
            delivery_home_2022_rough_wide.select(
                "hh_geog", "total_exc_missingresponse"
            ),
            on="hh_geog",
        )
        .with_columns(
            weighted_per_hhgeog_personday=(
                pl.col("weighted") / pl.col("total_exc_missingresponse")
            )
        )
        .drop("total_exc_missingresponse")
        .with_columns(
            delivery_type_rough=pl.col("delivery_type").str.split("_").list.get(1)
        )
    ),
    "delivery_type",
    sum_over_cols=["year", "hh_geog", "delivery_type_rough"],
).sort("hh_geog", "delivery_type")


In [ ]:
delivery_home_2022_rough_long

In [ ]:
delivery_home_2022_fine_long